# bgm_stable_audio

**Stable Audio 3 Medium** 으로 BGM 생성.

| | |
|---|---|
| 커널 | `sa3` |
| 출력 | `outputs/bgm_stable_audio/<시나리오>/` |
| 개수 | 시나리오당 페이즈 2 x 후보 3 = 6개 |

## 페이즈가 참조하는 이야기가 다르다

- `story` ← `source_story` (원작 설화, 사건 전)
- `case`  ← `adapted_story` (각색 사건, 사건 후)

둘을 같은 톤으로 만들면 사건 전부터 살인 분위기가 깔려 낙차가 사라진다.

## 사전 준비 (1회)

1. huggingface.co/stabilityai/stable-audio-3-medium 에서 **약관 동의**
2. Settings → Access Tokens 에서 read 토큰 발급
3. 아래 2번 셀에 토큰 입력

gated repo라 동의·토큰 없이는 다운로드 단계에서 401이 난다.


## 1. 커널 확인

설치는 `setup_kernels.ipynb` 가 한다. 여기서는 커널이 맞는지만 본다.
커널을 `sa3` 로 바꾸고 실행할 것.

In [ ]:
# ---------------------------------------------------------------------------
# 커널 확인. 설치는 setup_kernels.ipynb 가 담당한다.
#
# 이 셀에 pip install 을 추가하지 말 것. 노트북 4개는 서로 양립 불가능한
# torch/transformers 를 쓰기 때문에, 여기서 다시 깔면 커널 조합이 깨진다.
#   ace  torch 2.10.0+cu128 / transformers >=4.51,<4.58
#   sa3  torch 2.7.1 cu126  / transformers >=5.8
#   flux torch 2.8.0        / transformers ==4.56.1
# ---------------------------------------------------------------------------
import sys, importlib

KERNEL = "sa3"
NEED   = ["torch", "torchaudio", "soundfile", "stable_audio_3"]

print("python     :", sys.version.split()[0])
print("interpreter:", sys.executable)
print()

missing = []
for m in NEED:
    try:
        mod = importlib.import_module(m)
        print(f"OK   {m:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        missing.append(m)
        print(f"FAIL {m:16s} {type(e).__name__}: {e}")

assert not missing, (
    f"\n{missing} 를 import 하지 못했다.\n"
    f"커널을 '{KERNEL}' 로 바꿀 것 (Kernel > Change kernel).\n"
    f"'{KERNEL}' 커널이 목록에 없으면 setup_kernels.ipynb 를 먼저 실행한다.\n"
    f"여기서 pip install 로 때우지 말 것 — 버전 조합이 깨진다."
)
print("\n커널 OK")

## 2. 인증

Hugging Face 토큰. 저장소에 커밋하지 말 것.


In [ ]:
import os, pathlib

# 토큰을 노트북에 적지 않는다. 커밋되면 그대로 노출된다.
# 터미널에서:  export HF_TOKEN=hf_xxx   (커널을 띄우기 전에. 이미 떠 있으면 커널 재시작)
# 또는 파일로:  echo hf_xxx > ~/.hf_token
tok = os.environ.get("HF_TOKEN", "").strip()
if not tok:
    p = pathlib.Path.home() / ".hf_token"
    if p.exists():
        tok = p.read_text().strip()
        os.environ["HF_TOKEN"] = tok

assert tok.startswith("hf_"), (
    "HF_TOKEN 이 없다.\n"
    "  export HF_TOKEN=hf_...   후 커널 재시작\n"
    "  또는  echo hf_... > ~/.hf_token\n"
    "gated repo 라 토큰 없이는 401 이 난다."
)
print("토큰 설정됨:", tok[:6] + "…" + f" ({len(tok)}자)")

## 3. 설정


In [ ]:
import os, sys, json, pathlib, getpass, shutil

USER = getpass.getuser()

# ---- 이 노드에서 유효한 루트를 직접 찾는다 --------------------------------
# 학과 클러스터는 같은 저장소가 노드마다 다른 경로에 마운트된다.
#   로그인 노드(hpcmaster) : /data1/$USER   (= HOME)
#   계산 노드(hpc-stat1)   : /mnt/data1/$USER
# setup_kernels 로 만든 커널이면 PROJ_BASE 가 kernel.json 에 박혀 있고,
# Slurm 으로 돌리면 common.sh 가 환경변수로 넘겨준다. 둘 다 아니면 여기서 찾는다.
def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(f"쓸 수 있는 루트를 못 찾았다 (user={USER})")

# 계산 노드에는 $HOME 이 아예 없다. 두면 캐시/kernelspec 이 전부 깨진다.
if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT

BASE  = pathlib.Path(os.environ.get("PROJ_BASE") or f"{ROOT}/mystery")
MODEL = "bgm_stable_audio"
OUT_DIR = BASE / "outputs" / MODEL

# 캐시는 홈이 아니라 대용량 스토리지로. 홈 쿼터면 다운로드 중 os error 28 이 난다.
os.environ.setdefault("HF_HOME",      str(BASE / "hf_cache"))
os.environ.setdefault("TMPDIR",       str(BASE / "tmp"))
os.environ.setdefault("MPLCONFIGDIR", str(BASE / "mpl_cache"))

try:
    for p in (OUT_DIR, pathlib.Path(os.environ["HF_HOME"]),
              pathlib.Path(os.environ["TMPDIR"])):
        p.mkdir(parents=True, exist_ok=True)
except (PermissionError, OSError) as e:
    raise SystemExit(
        f"{BASE} 에 쓸 수 없다 -> {type(e).__name__}: {e}\n"
        f"  이 셀 맨 위에서 지정할 것:\n"
        f"      os.environ['PROJ_BASE'] = '/data1/{USER}/mystery'"
    )

# ---- 입력 JSON (scenarios_en/*.json 전부) ---------------------------------
# papermill 로 돌리면 cwd 가 /tmp 라 "." 만으로는 못 찾는다.
# PROJ_INPUT(환경변수) 을 먼저 보고, 없으면 후보들을 뒤진다.
def pick_input_dir():
    cands = []
    if os.environ.get("PROJ_INPUT"):
        cands.append(pathlib.Path(os.environ["PROJ_INPUT"]))
    cands += [pathlib.Path.cwd(), pathlib.Path.cwd().parent, BASE]
    for c in cands:
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    # 마지막 수단: ROOT 아래를 얕게 훑는다
    for c in pathlib.Path(ROOT).glob("*/*/"):
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    return None

IN_DIR = pick_input_dir()
if IN_DIR is None:
    raise SystemExit(
        "scenarios_en/*.json 을 못 찾았다.\n"
        f"  PROJ_INPUT = {os.environ.get('PROJ_INPUT', '(미설정)')}\n"
        f"  cwd        = {pathlib.Path.cwd()}\n"
        f"  이 셀 맨 위에서:  os.environ['PROJ_INPUT'] = '/data1/{USER}/<저장소>/Jaemin'"
    )

# scenario_id(파일 내부 필드, 예: "12_heungbu")를 키로 쓴다 — 파일명과 무관하게 안정적.
SCENARIO_FILES = sorted((IN_DIR / "scenarios_en").glob("*.json"))
assert SCENARIO_FILES, f"scenarios_en 에 json 이 없다: {IN_DIR / 'scenarios_en'}"

DATA = {}
for _f in SCENARIO_FILES:
    _d = json.load(open(_f, encoding="utf-8"))
    DATA[_d["scenario_id"]] = _d

for k in DATA:
    (OUT_DIR / k).mkdir(parents=True, exist_ok=True)


def out_path(scen, *parts):
    """OUT_DIR/<시나리오>/<...> 경로를 만들고 돌려준다."""
    p = OUT_DIR / scen
    for x in parts[:-1]:
        p = p / x
    p.mkdir(parents=True, exist_ok=True)
    return p / parts[-1]


print("node    :", os.uname().nodename)
print("ROOT    :", ROOT)
print("BASE    :", BASE, f"(여유 {shutil.disk_usage(BASE).free/1e9:.0f} GB)")
print("IN_DIR  :", IN_DIR)
print("HF_HOME :", os.environ["HF_HOME"])
print("OUT_DIR :", OUT_DIR)
print(f"시나리오 {len(DATA)}개:")
for k, d in DATA.items():
    print(f"  {k:24s} 장소 {len(d['places'])} / 인물 {len(d['cast'])}")


In [ ]:
import gc, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print("torch  :", torch.__version__)
print("device :", DEVICE)
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU    :", p.name, "|", round(p.total_memory / 1e9, 1), "GB",
          "| compute capability", f"{p.major}.{p.minor}")
    if (p.major, p.minor) < (8, 9):
        print("  주의: FP8 연산 미지원 세대. FP8 가중치를 요구하는 모델은 못 돌린다.")
else:
    print("  GPU가 없다. 로그인 노드이거나 GPU 할당이 안 된 상태.")


def free():
    """모델을 내린 뒤 호출. 커널을 종료하는 것이 가장 확실하다."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        print(f"할당된 VRAM: {used:.2f} GB")


## 5. 모델 초기화

In [ ]:
SR = 44100  # 기본 샘플레이트 (모델이 알려주면 아래에서 덮어씀)

from stable_audio_3 import StableAudioModel
import soundfile as sf
import numpy as np
import torch

sa3 = StableAudioModel.from_pretrained("medium", device=DEVICE)

# 저장 샘플레이트는 모델이 알려주면 그걸 따른다. 헤더에 틀린 값을 쓰면
# 에러 없이 재생 속도와 음높이가 어긋난다.
SR_OUT = int(getattr(sa3, "sample_rate", SR))
print("저장 샘플레이트:", SR_OUT, "(모델이 보고한 값)" if hasattr(sa3, "sample_rate") else "(모델이 안 알려줘 기본값 사용)")


## 6. 큐 조립

In [ ]:
# ---------------------------------------------------------------------------
# 확장 BGM 큐 파일럿 — 12_흥부전 하나만.
# cues(7) + place_cues(6장소x2상태=12) + character_themes(5) + event_cues(12)
# = 36개.
#
# json/ 의 bgm.*.prompt 필드는 시나리오팀이 작곡 참고용으로 쓴 문서라 실제로는
# 한국어 음악 용어(예: "상행 4도 도약 뒤 순차 하행", "시김새")가 영어 문장 사이에
# 섞여 있고, 모든 큐마다 같은 "IMPORTANT — all tracks share ONE main theme..."
# 블록이 반복돼 프롬프트가 200단어를 넘는다. stable-audio-3 의 T5Gemma 텍스트
# 인코더는 max_length=256 토큰에서 뒷부분을 그대로 잘라버리는데, 실측해보니
# 실제로 악기 지정(instrumentation: ...)이 잘려나가는 프롬프트가 있었다.
# 그래서 bgm.*.prompt 를 그대로 쓰지 않고, 구조화된 필드(bpm/dynamics/
# instruments/mood/when)를 직접 영어로 번역해 짧고 태그 중심인 프롬프트를
# 새로 만든다 (build_prompt, 아래).
# ---------------------------------------------------------------------------
import re, json as _json

PILOT_ID = "12_heungbu"
PILOT_JSON = IN_DIR / "json" / "12_흥부전.json"
assert PILOT_JSON.exists(), f"파일럿 원본을 못 찾음: {PILOT_JSON}"
_raw = _json.load(open(PILOT_JSON, encoding="utf-8"))
bgm_full = _raw["bgm"]


def parse_seconds(form_text, default=60):
    m = re.search(r"(\d+)\s*[~-]\s*(\d+)\s*초", form_text or "")
    if m:
        return (int(m.group(1)) + int(m.group(2))) / 2
    m = re.search(r"(\d+)\s*초", form_text or "")
    return int(m.group(1)) if m else default


jobs = []
for cid, c in bgm_full["cues"].items():
    jobs.append(("cues", cid, c))
for pid, p in bgm_full["place_cues"].items():
    for state in ("pre_murder", "post_murder"):
        jobs.append(("place_cues", f"{pid}_{state}", p[state]))
for cid, c in bgm_full["character_themes"].items():
    jobs.append(("character_themes", cid, c))
for eid, e in bgm_full["event_cues"].items():
    jobs.append(("event_cues", eid, e))

print(f"{PILOT_ID} 확장 BGM: {len(jobs)}개 큐")
for cat, cid, c in jobs:
    secs = parse_seconds(c.get("form"))
    tag = "one-shot" if "one-shot" in (c.get("form") or "") else "loop"
    print(f"  [{cat}/{cid}] {secs:.0f}s ({tag})")


INSTR_EN = {
    "대금": "daegeum bamboo flute", "해금": "haegeum two-string fiddle",
    "거문고": "geomungo six-string zither", "아쟁": "ajaeng bowed zither",
    "장구": "janggu hourglass drum", "소리북": "sori-buk barrel drum",
    "박(拍)": "bak wooden clapper", "꽹과리": "kkwaenggwari small gong",
    "목어": "moktak wooden fish block", "프살테리": "plucked psaltery-style zither",
    "징": "jing large gong", "리코더": "wooden recorder flute",
}

def instr_en(ko_list):
    return ", ".join(INSTR_EN.get(k, k) for k in ko_list)

PLACE_EN = {
    "PC": "Nolbu's reception room", "P1": "Nolbu's wife's room",
    "P2": "Heungbu's thatched hut", "P3": "the steward's room",
    "P4": "the neighboring laborer's house", "P5": "the herbalist's outdoor hut",
}
PLACE_UNIQUE_PRE = {"PC": "still nothing has happened here — a calm about to collapse"}
PLACE_UNIQUE_POST = {"PC": "where the body was laid; hushed stillness and low-register pressure"}

PERSONA_EN = {"가련": "pitiable", "표준": "composed, neutral", "음험": "sinister"}

CUE_EN = {
    "intro":         ("the original tale's narration ('once upon a time...')", "bright, fable-like, nostalgic"),
    "discovery":     ("the moment the body is found — the story's turn", "shock, sudden stillness, dissonance, a hard stop"),
    "investigation": ("exploring the scene and location", "calm tension, focused observation"),
    "interrogation": ("questioning a suspect", "a character's inner psychology, subtle pressure"),
    "climax":        ("the decisive clue is revealed / the final accusation", "mounting conviction and dread"),
    "reveal":        ("the true culprit is revealed, a confession", "tragic catharsis, the sorrow of the motive"),
    "ending":        ("the case is closed, epilogue", "resignation and a lingering feeling, a return to the original tale"),
}

EVENT_EN = {
    "round_start":     ("a new round begins, investigation resumes", "time closing in again"),
    "clue_found":      ("a general clue is revealed", "small thrill of discovery, a brief rise"),
    "clue_forensic":   ("a coroner's report or body-related clue is revealed", "the cold weight of fact"),
    "clue_decisive":   ("the decisive clue is revealed, the door to the truth", "everything clicking into place — conviction and a shiver"),
    "red_herring":     ("a misleading clue (red herring)", "plausible but slightly off, a cadence resolved a half-step wrong"),
    "search_hit":      ("a search finds the right word", "the small satisfaction of finding the answer"),
    "search_miss":     ("a search returns nothing", "a false lead, two notes falling weakly"),
    "accusation":      ("the culprit-naming stage, final deduction", "the detective's moment when the pieces click together"),
    "verdict_correct": ("culprit, weapon, and motive are all correct", "a victory tinged with relief and bitterness"),
    "verdict_wrong":   ("the accusation fails", "crumbling certainty, the air cooling fast"),
    "life_lost":       ("one life or chance is used up", "one irreversible blow"),
    "time_low":        ("few interrogation turns remain", "a hurrying pulse"),
}

CHAR_WHEN_EN = {
    "C1": "questioning Heungbu", "C2": "questioning Nolbu's wife",
    "C3": "questioning the steward Maeho", "C4": "questioning the neighboring laborer",
    "C5": "questioning the herbalist old woman",
}

MAIN_THEME_NOTE = ("variation on this scenario's one shared main motif: a rising 4th leap "
                    "then a stepwise descent, lead instrument daegeum, key center B minor-ish; "
                    "keep the same melodic contour, vary only tempo, instrumentation and mood.")


def build_prompt(cat, cid, c):
    """json/ 의 bgm.*.prompt(한국어 섞인 원문) 대신, 구조화된 필드를 영어로
    번역해 짧고 태그 중심인 프롬프트를 만든다. 위 헤더 주석의 truncation 이슈 참고."""
    instruments = instr_en(c.get("instruments", []))
    bpm = c.get("bpm")
    dyn = c.get("dynamics")

    if cat == "cues":
        when_en, mood_en = CUE_EN[cid]
        variation = "this is the scenario's main theme statement." if cid == "intro" else MAIN_THEME_NOTE
    elif cat == "event_cues":
        when_en, mood_en = EVENT_EN[cid]
        variation = MAIN_THEME_NOTE
    elif cat == "place_cues":
        if cid.endswith("_pre_murder"):
            pid, state = cid[: -len("_pre_murder")], "pre_murder"
        else:
            pid, state = cid[: -len("_post_murder")], "post_murder"
        place = PLACE_EN[pid]
        if state == "pre_murder":
            mood_en = PLACE_UNIQUE_PRE.get(pid, f"peaceful, everyday air of {place}, before anything has happened")
            when_en = f"exploring {place} (before the murder)"
        else:
            mood_en = PLACE_UNIQUE_POST.get(pid, "the same place turned cold — warmth drained, suspicion settling in")
            when_en = f"exploring {place} (after the murder)"
        variation = MAIN_THEME_NOTE
    elif cat == "character_themes":
        when_en = CHAR_WHEN_EN[cid]
        persona_en = PERSONA_EN.get(c.get("persona_flavor"), c.get("persona_flavor"))
        mood_en = f"a {persona_en} character's inner state; a layer is added as pressure rises"
        variation = MAIN_THEME_NOTE
    else:
        raise ValueError(cat)

    return (
        f"instrumental cue for a Korean traditional detective mystery game. "
        f"instrumentation: {instruments}, Korean gyemyeonjo pentatonic mode. "
        f"scene: {when_en}. mood: {mood_en}. "
        f"tempo: {bpm} BPM, dynamics {dyn}. {variation} "
        f"no vocals, no lyrics. avoid: Western orchestral brass, drum kit, synth pads."
    )


# 실측 확인(2026-08-24): cfg_scale=7 에서는 negative_prompt 가 실제로 출력에
# 영향을 준다(cfg_scale=1 에서는 CFG 공식상 무효 — 그래서 여기선 항상 7을 쓴다).
BASE_NEGATIVE = ("distorted, harsh dissonance, atonal noise, static, glitch, "
                  "poor audio quality, muddy recording, out of tune, screeching feedback")
CALM_NEGATIVE_EXTRA = ", scary, horror, ominous dread, frightening, unsettling"

# 원래 밝거나 평온해야 하는 큐만 scary/horror 계열도 추가로 배제한다.
# discovery/climax/reveal/interrogation/accusation 등은 긴장감이 의도된
# 부분이라 여기 넣으면 오히려 밋밋해진다.
LIGHT_CUES = {("cues", "intro"), ("cues", "ending"),
              ("event_cues", "clue_found"), ("event_cues", "search_hit")}


def build_negative(cat, cid):
    is_light = (cat, cid) in LIGHT_CUES or (cat == "place_cues" and cid.endswith("_pre_murder"))
    return BASE_NEGATIVE + (CALM_NEGATIVE_EXTRA if is_light else "")


## 7. 생성

In [ ]:
import soundfile as sf
import numpy as np

N_EXPANDED = 1

for cat, cid, c in jobs:
    secs = parse_seconds(c.get("form"), default=60)
    outdir = OUT_DIR / PILOT_ID / cat
    outdir.mkdir(parents=True, exist_ok=True)
    out_file = outdir / f"{cid}_0.wav"
    if out_file.exists():
        print("   (이미 있음, 건너뜀)", cat, cid)
        continue

    audio = sa3.generate(prompt=build_prompt(cat, cid, c), negative_prompt=build_negative(cat, cid), duration=secs, steps=8, cfg_scale=3.0, seed=0)
    wav = audio[0]
    if isinstance(wav, torch.Tensor):
        wav = wav.T.float().cpu().numpy()
    else:
        wav = np.asarray(wav).T
    peak = np.abs(wav).max()
    if peak > 0:
        target_peak = 10 ** (-1.0 / 20)  # -1 dBFS, ACE-Step과 동일 기준
        wav = wav / peak * target_peak
    wav = np.clip(wav, -1.0, 1.0)
    sf.write(out_file, wav, SR_OUT)
    print("OK", cat, cid)

print("\n확장 BGM 저장 위치:", OUT_DIR / PILOT_ID)


## 8. 정리

In [ ]:
del sa3
free()
